# Chapter 31 — APIs, scheduled refresh, report generation, and external data

**Book role:** Advanced workflows  
**Suggested book space:** about 7 pages after adding diagrams, annotated screenshots, worked examples, and review exercises.  
**How to study:** read the explanations, run one code cell at a time, change the inputs, then complete the lab without copying the solution first.

> **Execution note:** Tkinter opens native desktop windows. Run GUI cells in a local Python/Jupyter session with a working display, not a headless notebook server. Close each window before moving to the next GUI example. Do not use **Run All** on the complete book: some cells open windows, read or write files, or demonstrate database operations.

## Learning outcomes

- Consume a web API from a data service.
- Separate network timeouts, HTTP errors, and malformed payloads.
- Use background work and caching appropriately.
- Document rate limits, provenance, and fallback behavior.

## 1. External data brings new failure modes

An API can be unavailable, slow, rate-limited, or return a valid HTTP response with an unexpected JSON schema. Handle timeouts, HTTP status errors, JSON decoding, and schema validation distinctly. Do not make a network request during module import or in the Tk window constructor; make it an explicit refresh action and keep slow work out of the main event loop.

Use a clear user-agent or API key when required by the provider’s terms, store secrets outside source code, respect rate limits, and provide attribution when mandated. Record the request date/time, relevant query parameters, and source version where possible. Do not call a public API in every widget trace or poll aggressively without a justified refresh policy.

### Worked example 1: Validate an external JSON payload

A robust implementation would make the request with connect/read timeouts, check status codes, and validate the provider’s schema version. The payload is inline so the example can run offline.

In [ ]:
from datetime import datetime, timezone
import pandas as pd

payload = {
    "generated_at": "2026-10-09T08:00:00Z",
    "results": [
        {"region": "North", "revenue": 125000.0},
        {"region": "West", "revenue": 98000.0},
    ],
}
frame = pd.DataFrame(payload.get("results", []))
required = {"region", "revenue"}
missing = required - set(frame.columns)
if missing:
    raise ValueError(f"API result is missing columns: {sorted(missing)}")
frame["revenue"] = pd.to_numeric(frame["revenue"], errors="raise")
source_time = pd.to_datetime(payload["generated_at"], utc=True, errors="raise")
print(frame)
print("Source timestamp:", source_time)
print("Retrieved at UTC:", datetime.now(timezone.utc).isoformat())

## 2. Caching and refresh semantics

Caching reduces repeated network requests and helps users work when a source is temporarily unavailable. A cache needs a timestamp, source identity, and expiry policy; otherwise a stale response may be mistaken for live data. Display “last refreshed at” and offer Refresh Now where useful. A failed refresh should leave the previous result visible with a warning rather than silently clear valid data unless safety requirements demand invalidation.

Some APIs paginate results, apply daily quotas, or limit maximum record count. Load all required pages responsibly and check for truncation. If only partial data loaded, communicate that state explicitly. For long-running pulls, provide progress based on pages or records processed and support a user-controlled cancellation policy where technically possible.

### Worked example 2: A bounded HTTP request pattern

A timeout tuple separates connection establishment and reading. The function is synchronous and belongs in a worker when used by a GUI.

In [ ]:
import requests


def get_json(url: str, params: dict | None = None) -> dict:
    response = requests.get(url, params=params, timeout=(3.05, 15))
    response.raise_for_status()
    content_type = response.headers.get("Content-Type", "")
    if "json" not in content_type.lower():
        raise ValueError("Expected a JSON response")
    payload = response.json()
    if not isinstance(payload, dict):
        raise ValueError("Expected a JSON object at the response root")
    return payload

# Example call intentionally omitted: use a documented API and comply with its authentication/rate-limit rules.

## 3. Reporting and reproducibility

An exported report should contain the analysis period, applied filters, data-source label, refresh timestamp, definitions for metrics, and known data-quality caveats. This metadata makes a workbook or PDF interpretable after it has been separated from the application. Avoid presenting a refreshed timestamp as if it were the observation date; data arrival time and business event time are different.

A scheduled refresh may be suitable for a local report cache or service, but avoid multiple overlapping refreshes. Keep the refresh schedule configurable and explicit. If the source needs a browser login, terms-of-service restrictions, or secure server-side secrets, a desktop client may not be the right place to implement the connector.

## Guided lab: Lab: resilient refresh workflow

Create a refresh service that requests a paginated dataset, validates each page, normalizes it, and records timestamps/source metadata. Run it in a worker, cache the last successful result, and indicate stale data after a failed refresh. Test timeout, HTTP error, malformed JSON, missing required fields, empty results, and partial pagination.

## Review questions and practice

1. Why should API calls not run inside a widget constructor?
2. What is the difference between business event time and refresh time?
3. How can caching mislead users if it has no timestamp?
4. Why must partial API pagination be visible?

## Further reading

- requests documentation: https://requests.readthedocs.io/
- HTTP semantics: https://developer.mozilla.org/en-US/docs/Web/HTTP

## Notebook checkpoint

Before moving on, write a short note in your own words explaining one design decision from this chapter, one failure mode to test, and one way the idea will be used in a data-analysis application.